# 02 - Data Preprocessing
## Text Cleaning, Transformation & Train-Test Split

**Objective:** Clean text data, handle missing values, and prepare train/test sets.

In [1]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

True

## 1. Load Data

In [2]:
df = pd.read_csv('../data/raw/WELFake_Dataset.csv')
print(f'Original shape: {df.shape}')
df.head()

Original shape: (72134, 4)


,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


## 2. Drop Unnecessary Columns

In [3]:
# Drop serial number column (first column)
if 'Unnamed: 0' in df.columns:
    df.drop(columns=['Unnamed: 0'], inplace=True)

print(f'Columns after dropping: {df.columns.tolist()}')
print(f'Shape: {df.shape}')

Columns after dropping: ['title', 'text', 'label']
Shape: (72134, 3)


## 3. Handle Missing Values

In [4]:
print('Missing values before:')
print(df.isnull().sum())
print(f'\nTotal rows: {len(df)}')

Missing values before:
title    558
text      39
label      0
dtype: int64

Total rows: 72134


In [5]:
# Fill missing title with empty string
df['title'] = df['title'].fillna('')

# Fill missing text with empty string
df['text'] = df['text'].fillna('')

# Drop rows where both title and text are empty
df = df[~((df['title'] == '') & (df['text'] == ''))]

print('Missing values after:')
print(df.isnull().sum())
print(f'Shape after: {df.shape}')

Missing values after:
title    0
text     0
label    0
dtype: int64
Shape after: (72134, 3)


## 4. Combine Title + Text

In [6]:
# Combine title and text into one column for better classification
df['content'] = df['title'] + ' ' + df['text']
df['content'] = df['content'].str.strip()

print('Sample combined content:')
print(df['content'].iloc[0][:300])

Sample combined content:
LAW ENFORCEMENT ON HIGH ALERT Following Threats Against Cops And Whites On 9-11By #BlackLivesMatter And #FYF911 Terrorists [VIDEO] No comment is expected from Barack Obama Members of the #FYF911 or #FukYoFlag and #BlackLivesMatter movements called for the lynching and hanging of white people and cop


## 5. Text Cleaning Function

In [7]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))


def clean_text(text):
    """Clean and preprocess text."""
    # Lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Tokenize, remove stopwords, and lemmatize
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    
    return ' '.join(words)


# Test the function
sample = df['content'].iloc[0]
print('Before:', sample[:200])
print('\nAfter:', clean_text(sample)[:200])

Before: LAW ENFORCEMENT ON HIGH ALERT Following Threats Against Cops And Whites On 9-11By #BlackLivesMatter And #FYF911 Terrorists [VIDEO] No comment is expected from Barack Obama Members of the #FYF911 or #F

After: law enforcement high alert following threat cop white blacklivesmatter fyf terrorist video comment expected barack obama member fyf fukyoflag blacklivesmatter movement called lynching hanging white pe


In [8]:
# Apply cleaning to all rows
print('Cleaning text... (this may take a few minutes)')
df['cleaned_content'] = df['content'].apply(clean_text)
print('Done!')

print(f'\nSample cleaned text:')
print(df['cleaned_content'].iloc[0][:300])

Cleaning text... (this may take a few minutes)
Done!

Sample cleaned text:
law enforcement high alert following threat cop white blacklivesmatter fyf terrorist video comment expected barack obama member fyf fukyoflag blacklivesmatter movement called lynching hanging white people cop encouraged others radio show tuesday night turn tide kill white people cop send message kil


## 6. Check Cleaned Data

In [9]:
# Check for empty strings after cleaning
empty_count = (df['cleaned_content'] == '').sum()
print(f'Empty rows after cleaning: {empty_count}')

# Drop empty rows
df = df[df['cleaned_content'] != '']
print(f'Final shape: {df.shape}')

Empty rows after cleaning: 54
Final shape: (72080, 5)


## 7. Train-Test Split

In [10]:
X = df['cleaned_content']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set:     {X_test.shape[0]} samples')
print(f'\nTrain label distribution:')
print(y_train.value_counts(normalize=True) * 100)
print(f'\nTest label distribution:')
print(y_test.value_counts(normalize=True) * 100)

Training set: 57664 samples
Test set:     14416 samples

Train label distribution:
label
1    51.404689
0    48.595311
Name: proportion, dtype: float64

Test label distribution:
label
1    51.401221
0    48.598779
Name: proportion, dtype: float64


In [12]:
# Save train and test sets
import os

# Create directory if it doesn't exist
os.makedirs('../artifacts/data_transformation/', exist_ok=True)

train_df = pd.DataFrame({'content': X_train, 'label': y_train})
test_df = pd.DataFrame({'content': X_test, 'label': y_test})

train_df.to_csv('../artifacts/data_transformation/train.csv', index=False)
test_df.to_csv('../artifacts/data_transformation/test.csv', index=False)

print('Train and test sets saved!')
print(f'Train: {train_df.shape}')
print(f'Test:  {test_df.shape}')

Train and test sets saved!
Train: (57664, 2)
Test:  (14416, 2)


## 8. Summary

Steps performed:
1. Dropped serial number column
2. Filled missing title/text with empty string
3. Combined title + text into `content`
4. Cleaned text: lowercase, remove URLs/HTML/special chars, remove stopwords, lemmatize
5. Split into 80% train / 20% test (stratified)
6. Saved to `artifacts/data_transformation/`